In [1]:
import os

import pandas as pd
import string

## Keras
from keras.layers import Dense, Flatten, SimpleRNN
from keras import Sequential

## nltk
import nltk
from nltk.corpus import stopwords, re
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

## sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv(r"C:\Users\Krish\Downloads\Movie Sentiment Analysis\data\IMDB Dataset.csv")
df

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Krish\\Downloads\\Movie Sentiment Analysis\\data\\IMDB Dataset.csv'

In [ ]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [ ]:
df.shape

(50000, 2)

## Preprocessing Methods

In [ ]:
def rem_stopwords(text):
    words = stopwords.words('english')

    return " ".join(
        word for word in text.split() if word.lower() not in words
    )

def to_lower(text):
    return "".join(
        text.lower()
    )

def rem_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'', text)

def rem_punctuations(text):
    exclude = string.punctuation
    return text.translate(str.maketrans('', '', exclude))

nltk.download('punkt')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def lemmatization(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t.isalpha()]
    return " ".join(tokens)

def tokenize(text):
    tokenizer = word_tokenize(text)
    return tokenizer
    

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Krish\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Krish\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
df_rem = df
df_rem["review"] = df["review"].apply(to_lower)

df_rem["review"] = df["review"].apply(rem_stopwords)

In [ ]:
df_rem["review"] = df_rem["review"].apply(rem_tags)
df_rem["review"] = df_rem["review"].apply(rem_punctuations)

In [ ]:
print(df_rem["review"])

0        one reviewers mentioned watching 1 oz episode ...
1        wonderful little production the filming techni...
2        thought wonderful way spend time hot summer we...
3        basically theres family little boy jake thinks...
4        petter matteis love time money visually stunni...
                               ...                        
49995    thought movie right good job creative original...
49996    bad plot bad dialogue bad acting idiotic direc...
49997    catholic taught parochial elementary schools n...
49998    going disagree previous comment side maltin on...
49999    one expects star trek movies high art fans exp...
Name: review, Length: 50000, dtype: str


In [ ]:
df_rem["review"] = df_rem["review"].apply(lemmatization)

In [ ]:
## Tokenizing
df_rem["review"]= df_rem["review"].apply(tokenize)

In [ ]:
df_rem["review"]

0        [one, reviewer, mentioned, watching, oz, episo...
1        [wonderful, little, production, the, filming, ...
2        [thought, wonderful, way, spend, time, hot, su...
3        [basically, there, family, little, boy, jake, ...
4        [petter, matteis, love, time, money, visually,...
                               ...                        
49995    [thought, movie, right, good, job, creative, o...
49996    [bad, plot, bad, dialogue, bad, acting, idioti...
49997    [catholic, taught, parochial, elementary, scho...
49998    [going, disagree, previous, comment, side, mal...
49999    [one, expects, star, trek, movie, high, art, f...
Name: review, Length: 50000, dtype: object

In [ ]:
df['clean_review'] = df['review'].apply(lambda tokens: ' '.join(tokens))
X = df['clean_review']
y = df_rem["sentiment"]

encode = LabelEncoder()
y = encode.fit_transform(y,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size= 0.2, stratify= y, random_state=42
)

In [62]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)  # transform, NOT fit_transform

In [63]:
model = Sequential()

model.add(SimpleRNN(120, activation = "relu", input_shape = (1,1)))
model.add(Flatten())
model.add(Dense(1, activation  = "sigmoid"))

model.compile(loss= "binary_crossentropy", optimizer= "adam", metrics=['accuracy'])

c:\Users\Krish\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [64]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_2 (SimpleRNN)        │ (None, 120)            │        14,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           121 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,761 (57.66 KB)

 Trainable params: 14,761 (57.66 KB)

 Non-trainable params: 0 (0.00 B)

In [65]:
model.fit(X_train_tfidf, y_train, validation_data= (X_test, y_test))

  87/1250 ━━━━━━━━━━━━━━━━━━━━ 24:54 1s/step - accuracy: 0.4855 - loss: 0.6933

KeyboardInterrupt: 

In [53]:
X_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1 stored elements and shape (1, 1)>

<class 'pandas.DataFrame'>
40000


,review
47808,"[caught, little, gem, totally, accident, back,..."
20154,"[cant, believe, let, movie, accomplish, favor,..."
43069,"[spoiler, alert, get, nerve, people, remake, a..."
19413,"[there, one, thing, learnt, watching, george, ..."
13673,"[remember, theater, review, said, horrible, we..."
